# 08. Support Vector Machines: Márgenes Máximos

**Nivel:** 🔴 Avanzado  
**Tiempo estimado:** 90 minutos  
**Prerequisitos:** [03. Regresión Logística](03-regresion-logistica.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Entender el concepto de margen máximo y support vectors
- Derivar la función objetivo del SVM y su forma dual
- Comprender e implementar el kernel trick
- Aplicar diferentes kernels (lineal, RBF, polynomial)
- Tunear el parámetro C para balancear margen y errores
- Usar SVMs en problemas de clasificación no lineal

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.svm import SVC, LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.datasets import make_classification, make_circles, make_moons, load_breast_cancer
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Importar utilidades
import sys
sys.path.append('../../shared/utils')
from visualization import plot_decision_boundary
from datasets import load_dataset

np.random.seed(42)
print("✅ Librerías importadas")

---
## 📌 1. Motivación: Más Allá de "Cualquier" Línea Separadora

### El Problema con Regresión Logística

Recuerda regresión logística: encuentra un hiperplano que separa las clases.

**Pero...** ¿qué pasa cuando hay MUCHOS hiperplanos posibles?

```
    🔴              |         
  🔴  🔴            |  🔵
    🔴              |    🔵
                    |  🔵
```

Todas estas líneas separan perfectamente:
- Línea A: muy pegada a los 🔴
- Línea B: muy pegada a los 🔵
- Línea C: en el medio

**Pregunta:** ¿Cuál generalizará mejor a nuevos datos?

**Respuesta:** La línea C, porque está **lo más lejos posible de ambas clases**.

### La Idea de Support Vector Machine

**Objetivo:** Encontrar el hiperplano con **margen máximo**.

```
                Margen
    🔴         ║     ║
  🔴  🔴       ║  |  ║  🔵
    🔴  ←------║  |  ║----→  🔵
             SV  |  SV    🔵
                 |
         Hiperplano óptimo
```

**Support Vectors (SV):** Los puntos más cercanos al hiperplano (justo en el margen).

**Insight clave:** Solo estos puntos importan para definir el clasificador.

### ¿Por qué "Support Vector"?

Imagina los puntos como pelotas:
- El hiperplano es una tabla
- Los support vectors son las pelotas que **sostienen** (support) la tabla
- Si mueves una pelota lejana, la tabla no se mueve
- Si mueves un support vector, ¡la tabla se mueve!

### El Kernel Trick: Magia Dimensional

**Problema:** ¿Qué hacer cuando los datos NO son linealmente separables?

```
      🔵              
   🔵  🔴  🔵
  🔵 🔴 🔴 🔴 🔵       No hay línea que separe
   🔵  🔴  🔵
      🔵
```

**Solución:** ¡Proyectar a una dimensión superior donde SÍ sean separables!

```
2D (no separable) → Transformación → 3D (separable con un plano)
```

**Kernel trick:** Hacer esto implícitamente sin calcular las coordenadas de alta dimensión.

### Aplicaciones Reales

- 📝 **Reconocimiento de escritura**: MNIST, dígitos manuscritos
- 🧬 **Bioinformática**: Clasificación de proteínas, predicción de estructura
- 📷 **Computer Vision**: Detección de objetos (antes de deep learning)
- 📄 **Text Classification**: Categorización de documentos, spam detection
- 💊 **Descubrimiento de fármacos**: Predicción de actividad molecular

### La Pregunta Guía

> **¿Cómo podemos trabajar en espacios de dimensión infinita sin calcular explícitamente las coordenadas?**

---
## 📊 2. Intuición Visual

In [ ]:
# Comparación: Regresión Logística vs SVM
# Generar datos linealmente separables
from sklearn.linear_model import LogisticRegression

X, y = make_classification(n_samples=100, n_features=2, n_redundant=0,
                          n_informative=2, n_clusters_per_class=1,
                          class_sep=2.0, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Entrenar modelos
lr = LogisticRegression()
svm = SVC(kernel='linear', C=1.0)

lr.fit(X_train, y_train)
svm.fit(X_train, y_train)

# Crear grid
h = 0.02
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Predicciones
Z_lr = lr.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
Z_svm = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Regresión Logística
axes[0].contourf(xx, yy, Z_lr, alpha=0.3, cmap='RdBu')
axes[0].scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='RdBu', edgecolors='k')
axes[0].set_title(f'Regresión Logística\nAcc: {lr.score(X_test, y_test):.3f}', fontsize=14)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# SVM
axes[1].contourf(xx, yy, Z_svm, alpha=0.3, cmap='RdBu')
axes[1].scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='RdBu', edgecolors='k')
# Highlight support vectors
axes[1].scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1],
               s=200, facecolors='none', edgecolors='green', linewidths=2,
               label='Support Vectors')
axes[1].set_title(f'SVM (Margen Máximo)\nAcc: {svm.score(X_test, y_test):.3f}', fontsize=14)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n💡 Observaciones:")
print(f"   • SVM tiene {len(svm.support_vectors_)} support vectors (círculos verdes)")
print(f"   • Solo estos puntos definen la frontera de decisión")
print(f"   • El margen de SVM es máximo (más robusto)")

In [ ]:
# Visualizar el efecto del kernel en datos no lineales
# Generar datos circulares (no linealmente separables)
X_circles, y_circles = make_circles(n_samples=200, noise=0.1, factor=0.3, random_state=42)

# Entrenar SVMs con diferentes kernels
kernels = ['linear', 'poly', 'rbf']
svms = {}

for kernel in kernels:
    svm_model = SVC(kernel=kernel, gamma='auto')
    svm_model.fit(X_circles, y_circles)
    svms[kernel] = svm_model

# Crear grid
x_min, x_max = X_circles[:, 0].min() - 0.5, X_circles[:, 0].max() + 0.5
y_min, y_max = X_circles[:, 1].min() - 0.5, X_circles[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (kernel, svm_model) in enumerate(svms.items()):
    Z = svm_model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    axes[idx].contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    axes[idx].scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, 
                     cmap='RdBu', edgecolors='k')
    axes[idx].scatter(svm_model.support_vectors_[:, 0], 
                     svm_model.support_vectors_[:, 1],
                     s=200, facecolors='none', edgecolors='green', linewidths=2)
    axes[idx].set_title(f'Kernel: {kernel}\nAcc: {svm_model.score(X_circles, y_circles):.3f}',
                       fontsize=14)
    axes[idx].set_xlabel('Feature 1')
    axes[idx].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

print("\n💡 Observaciones:")
print("   • Kernel lineal falla (datos no son linealmente separables)")
print("   • Kernel polynomial puede capturar curvas")
print("   • Kernel RBF (Radial Basis Function) es muy flexible")

---
## 🧮 3. Fundamentos Matemáticos

### 📖 Notación

| Símbolo | Significado |
|---------|-------------|
| $\mathbf{w}$ | Vector de pesos (normal al hiperplano) |
| $b$ | Bias (desplazamiento del origen) |
| $\mathbf{x}_i$ | Ejemplo $i$ |
| $y_i \in \{-1, +1\}$ | Etiqueta de clase |
| $\alpha_i$ | Multiplicadores de Lagrange |
| $\xi_i$ | Variables de holgura (slack) |
| $C$ | Parámetro de regularización |

---

## 3.1 SVM Lineal: Margen Duro (Hard Margin)

### El Hiperplano Separador

Un hiperplano en $\mathbb{R}^n$ se define como:
$$
\mathbf{w}^T \mathbf{x} + b = 0 \tag{1}
$$

**Regla de decisión:**
$$
f(\mathbf{x}) = \text{sign}(\mathbf{w}^T \mathbf{x} + b) \tag{2}
$$

### Distancia de un Punto al Hiperplano

La distancia de $\mathbf{x}_i$ al hiperplano es:
$$
\text{distancia} = \frac{|\mathbf{w}^T \mathbf{x}_i + b|}{\|\mathbf{w}\|} \tag{3}
$$

### Margen

Queremos que todos los puntos estén correctamente clasificados con margen:
$$
y_i(\mathbf{w}^T \mathbf{x}_i + b) \geq 1, \quad \forall i \tag{4}
$$

El **margen geométrico** es:
$$
\gamma = \frac{2}{\|\mathbf{w}\|} \tag{5}
$$

**Objetivo:** Maximizar $\gamma$ = Minimizar $\|\mathbf{w}\|$

### Problema de Optimización (Primal)

$$
\begin{align}
\min_{\mathbf{w}, b} \quad & \frac{1}{2} \|\mathbf{w}\|^2 \tag{6} \\
\text{s.t.} \quad & y_i(\mathbf{w}^T \mathbf{x}_i + b) \geq 1, \quad i = 1, ..., n \tag{7}
\end{align}
$$

Este es un problema de **optimización cuadrática** con restricciones lineales.

---

## 3.2 Forma Dual (La Magia del Kernel)

Usando multiplicadores de Lagrange, el problema dual es:

$$
\begin{align}
\max_{\alpha} \quad & \sum_{i=1}^{n} \alpha_i - \frac{1}{2} \sum_{i=1}^{n} \sum_{j=1}^{n} \alpha_i \alpha_j y_i y_j \mathbf{x}_i^T \mathbf{x}_j \tag{8} \\
\text{s.t.} \quad & \alpha_i \geq 0, \quad \sum_{i=1}^{n} \alpha_i y_i = 0 \tag{9}
\end{align}
$$

**Solución:**
$$
\mathbf{w} = \sum_{i=1}^{n} \alpha_i y_i \mathbf{x}_i \tag{10}
$$

**Predicción:**
$$
f(\mathbf{x}) = \text{sign}\left(\sum_{i=1}^{n} \alpha_i y_i \mathbf{x}_i^T \mathbf{x} + b\right) \tag{11}
$$

**Observación clave:** Solo aparece $\mathbf{x}_i^T \mathbf{x}_j$ (producto interno).

**Support Vectors:** Muestras con $\alpha_i > 0$ (típicamente pocas).

---

## 3.3 Soft Margin SVM (Datos No Separables)

En la realidad, los datos raramente son perfectamente separables. Introducimos **variables de holgura** $\xi_i$:

$$
\begin{align}
\min_{\mathbf{w}, b, \xi} \quad & \frac{1}{2} \|\mathbf{w}\|^2 + C \sum_{i=1}^{n} \xi_i \tag{12} \\
\text{s.t.} \quad & y_i(\mathbf{w}^T \mathbf{x}_i + b) \geq 1 - \xi_i \tag{13} \\
& \xi_i \geq 0 \tag{14}
\end{align}
$$

**Interpretación:**
- $C$ pequeño: margen grande, permite más errores (underfitting)
- $C$ grande: margen pequeño, penaliza errores fuertemente (overfitting)

**Forma dual:**
$$
\begin{align}
\max_{\alpha} \quad & \sum_{i=1}^{n} \alpha_i - \frac{1}{2} \sum_{i=1}^{n} \sum_{j=1}^{n} \alpha_i \alpha_j y_i y_j \mathbf{x}_i^T \mathbf{x}_j \tag{15} \\
\text{s.t.} \quad & 0 \leq \alpha_i \leq C, \quad \sum_{i=1}^{n} \alpha_i y_i = 0 \tag{16}
\end{align}
$$

---

## 3.4 El Kernel Trick

**Idea:** Reemplazar $\mathbf{x}_i^T \mathbf{x}_j$ por $K(\mathbf{x}_i, \mathbf{x}_j)$ donde:

$$
K(\mathbf{x}_i, \mathbf{x}_j) = \phi(\mathbf{x}_i)^T \phi(\mathbf{x}_j) \tag{17}
$$

$\phi: \mathbb{R}^n \to \mathbb{R}^m$ es una transformación a un espacio de mayor dimensión.

**Magia:** ¡Podemos calcular $K$ sin conocer explícitamente $\phi$!

### Kernels Populares

#### 1. Kernel Lineal
$$
K(\mathbf{x}_i, \mathbf{x}_j) = \mathbf{x}_i^T \mathbf{x}_j \tag{18}
$$

#### 2. Kernel Polynomial
$$
K(\mathbf{x}_i, \mathbf{x}_j) = (\mathbf{x}_i^T \mathbf{x}_j + c)^d \tag{19}
$$

Ejemplo ($d=2$, $\mathbf{x} \in \mathbb{R}^2$):
$$
\phi([x_1, x_2]) = [x_1^2, \sqrt{2}x_1x_2, x_2^2, \sqrt{2c}x_1, \sqrt{2c}x_2, c] \tag{20}
$$

#### 3. Kernel RBF (Gaussian)
$$
K(\mathbf{x}_i, \mathbf{x}_j) = \exp\left(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2\right) \tag{21}
$$

Donde $\gamma = \frac{1}{2\sigma^2}$.

**Propiedad increíble:** ¡RBF corresponde a un espacio de dimensión INFINITA!

**Interpretación de $\gamma$:**
- $\gamma$ pequeño: influencia de cada punto es amplia (suavizado)
- $\gamma$ grande: influencia solo local (más flexible, puede overfit)

### Ejemplo Numérico

**Datos 1D:** $x_1 = 1$, $x_2 = 2$

**Kernel lineal:**
$$
K(1, 2) = 1 \times 2 = 2
$$

**Kernel polynomial ($d=2$, $c=0$):**
$$
K(1, 2) = (1 \times 2)^2 = 4
$$

**Kernel RBF ($\gamma=1$):**
$$
K(1, 2) = \exp(-1 \times (1-2)^2) = e^{-1} \approx 0.368
$$

---

## 3.5 Condiciones de Karush-Kuhn-Tucker (KKT)

En el óptimo, se cumplen:

$$
\begin{align}
\alpha_i [y_i(\mathbf{w}^T \mathbf{x}_i + b) - 1 + \xi_i] &= 0 \tag{22} \\
(C - \alpha_i) \xi_i &= 0 \tag{23}
\end{align}
$$

**Implicaciones:**

1. Si $\alpha_i = 0$: $\mathbf{x}_i$ no es support vector
2. Si $0 < \alpha_i < C$: $\mathbf{x}_i$ está en el margen ($\xi_i = 0$)
3. Si $\alpha_i = C$: $\mathbf{x}_i$ viola el margen ($\xi_i > 0$)

In [ ]:
# Demostración del kernel trick
# Comparar producto interno directo vs kernel

def linear_kernel(x1, x2):
    return np.dot(x1, x2)

def polynomial_kernel(x1, x2, degree=2, c=1):
    return (np.dot(x1, x2) + c) ** degree

def rbf_kernel(x1, x2, gamma=1.0):
    return np.exp(-gamma * np.linalg.norm(x1 - x2) ** 2)

# Ejemplos
x1 = np.array([1.0, 2.0])
x2 = np.array([3.0, 4.0])

print("🔢 Valores de Kernel para x1=[1, 2], x2=[3, 4]:\n")
print(f"Kernel Lineal:     {linear_kernel(x1, x2):.4f}")
print(f"Kernel Polynomial (d=2): {polynomial_kernel(x1, x2, degree=2):.4f}")
print(f"Kernel Polynomial (d=3): {polynomial_kernel(x1, x2, degree=3):.4f}")
print(f"Kernel RBF (γ=0.1):  {rbf_kernel(x1, x2, gamma=0.1):.4f}")
print(f"Kernel RBF (γ=1.0):  {rbf_kernel(x1, x2, gamma=1.0):.4f}")
print(f"Kernel RBF (γ=10):   {rbf_kernel(x1, x2, gamma=10.0):.4f}")

print("\n💡 El kernel RBF da valores entre 0 y 1 (similitud)")
print("   γ grande → más sensible a distancias pequeñas")

---
## 💻 4. Implementación Desde Cero

In [ ]:
class SimpleSVM:
    """
    Implementación simplificada de SVM lineal (hard margin).
    
    Usa scipy.optimize para resolver el problema dual.
    Solo para propósitos educativos.
    
    Parameters:
    -----------
    C : float, default=1.0
        Parámetro de regularización (no usado en hard margin)
    """
    
    def __init__(self, C=1.0):
        self.C = C
        self.w = None
        self.b = None
        self.support_vectors = None
        self.support_vector_labels = None
        self.alphas = None
    
    def fit(self, X, y):
        """
        Entrena el SVM usando programación cuadrática.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
        y : array-like, shape (n_samples,)
            Debe ser {-1, +1}
        """
        n_samples, n_features = X.shape
        
        # Convertir y a {-1, +1} si es necesario
        y = np.where(y <= 0, -1, 1)
        
        # Calcular matriz de Gram (K_ij = y_i * y_j * x_i^T * x_j)
        K = np.zeros((n_samples, n_samples))
        for i in range(n_samples):
            for j in range(n_samples):
                K[i, j] = y[i] * y[j] * np.dot(X[i], X[j])
        
        # Problema dual: max sum(alpha) - 0.5 * sum(alpha_i * alpha_j * K_ij)
        # Lo convertimos a minimización: min -sum(alpha) + 0.5 * sum(...)
        
        def objective(alpha):
            """Función objetivo (negativo porque scipy minimiza)"""
            return 0.5 * np.dot(alpha, np.dot(K, alpha)) - np.sum(alpha)
        
        def constraint_eq(alpha):
            """Restricción de igualdad: sum(alpha_i * y_i) = 0"""
            return np.dot(alpha, y)
        
        # Restricciones
        constraints = {'type': 'eq', 'fun': constraint_eq}
        bounds = [(0, self.C) for _ in range(n_samples)]
        
        # Resolver
        alpha0 = np.zeros(n_samples)
        result = minimize(objective, alpha0, method='SLSQP',
                        bounds=bounds, constraints=constraints)
        
        self.alphas = result.x
        
        # Identificar support vectors (alpha > threshold pequeño)
        sv_threshold = 1e-5
        sv_indices = self.alphas > sv_threshold
        
        self.support_vectors = X[sv_indices]
        self.support_vector_labels = y[sv_indices]
        support_alphas = self.alphas[sv_indices]
        
        # Calcular w
        self.w = np.sum(support_alphas[:, np.newaxis] * 
                       self.support_vector_labels[:, np.newaxis] * 
                       self.support_vectors, axis=0)
        
        # Calcular b (promediar sobre support vectors)
        b_values = []
        for i in range(len(self.support_vectors)):
            b_i = self.support_vector_labels[i] - np.dot(self.w, self.support_vectors[i])
            b_values.append(b_i)
        self.b = np.mean(b_values)
        
        print(f"✅ Entrenamiento completado")
        print(f"   Support vectors: {len(self.support_vectors)} de {n_samples}")
        print(f"   w: {self.w}")
        print(f"   b: {self.b:.4f}")
        
        return self
    
    def predict(self, X):
        """
        Predice clases para X.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
        
        Returns:
        --------
        y_pred : array, shape (n_samples,)
        """
        return np.sign(np.dot(X, self.w) + self.b)
    
    def score(self, X, y):
        """Calcula accuracy"""
        y = np.where(y <= 0, -1, 1)
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

print("✅ Clase SimpleSVM definida")

In [ ]:
# Probar con datos simples
X_simple, y_simple = make_classification(n_samples=100, n_features=2, n_redundant=0,
                                        n_informative=2, n_clusters_per_class=1,
                                        class_sep=2.0, random_state=42)

# Convertir a {-1, +1}
y_simple = np.where(y_simple == 0, -1, 1)

X_train, X_test, y_train, y_test = train_test_split(X_simple, y_simple, 
                                                    test_size=0.3, random_state=42)

# Normalizar (importante para SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Entrenar nuestro SVM
svm_custom = SimpleSVM(C=1.0)
svm_custom.fit(X_train_scaled, y_train)

In [ ]:
# Evaluar
train_acc = svm_custom.score(X_train_scaled, y_train)
test_acc = svm_custom.score(X_test_scaled, y_test)

print(f"\n📊 Resultados:")
print(f"   Train Accuracy: {train_acc:.4f}")
print(f"   Test Accuracy: {test_acc:.4f}")

# Visualizar
plt.figure(figsize=(10, 6))
plt.scatter(X_train_scaled[:, 0], X_train_scaled[:, 1], c=y_train, cmap='RdBu', edgecolors='k')
plt.scatter(svm_custom.support_vectors[:, 0], svm_custom.support_vectors[:, 1],
           s=200, facecolors='none', edgecolors='green', linewidths=2,
           label='Support Vectors')

# Dibujar hiperplano
xlim = plt.gca().get_xlim()
w = svm_custom.w
b = svm_custom.b
x_line = np.linspace(xlim[0], xlim[1], 100)
y_line = -(w[0] * x_line + b) / w[1]

plt.plot(x_line, y_line, 'k-', linewidth=2, label='Hiperplano')
plt.plot(x_line, y_line + 1/w[1], 'k--', linewidth=1, label='Margen')
plt.plot(x_line, y_line - 1/w[1], 'k--', linewidth=1)

plt.xlabel('Feature 1 (normalizada)')
plt.ylabel('Feature 2 (normalizada)')
plt.title('SVM: Margen Máximo')
plt.legend()
plt.show()

print("\n💡 Los support vectors definen completamente el hiperplano")

---
## 🏭 5. Versión con Framework (Scikit-learn)

In [ ]:
# Cargar Breast Cancer dataset
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# IMPORTANTE: Siempre normalizar para SVM
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Entrenar SVMs con diferentes kernels
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
models = {}

print("🚀 Entrenando SVMs con diferentes kernels...\n")

for kernel in kernels:
    svm = SVC(kernel=kernel, C=1.0, gamma='scale', random_state=42)
    svm.fit(X_train_scaled, y_train)
    models[kernel] = svm

# Comparar resultados
print("="*70)
print(f"{'Kernel':<15} {'Train Acc':<15} {'Test Acc':<15} {'# SVs':<15}")
print("="*70)

for kernel, svm in models.items():
    train_acc = svm.score(X_train_scaled, y_train)
    test_acc = svm.score(X_test_scaled, y_test)
    n_sv = len(svm.support_vectors_)
    print(f"{kernel:<15} {train_acc:<15.4f} {test_acc:<15.4f} {n_sv:<15}")

print("="*70)

In [ ]:
# Efecto del parámetro C
C_values = [0.001, 0.01, 0.1, 1, 10, 100]
train_scores = []
test_scores = []
n_svs = []

for C in C_values:
    svm = SVC(kernel='rbf', C=C, gamma='scale', random_state=42)
    svm.fit(X_train_scaled, y_train)
    train_scores.append(svm.score(X_train_scaled, y_train))
    test_scores.append(svm.score(X_test_scaled, y_test))
    n_svs.append(len(svm.support_vectors_))

# Visualizar
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=[str(c) for c in C_values],
    y=train_scores,
    mode='lines+markers',
    name='Train Accuracy',
    line=dict(color='blue', width=2)
))

fig.add_trace(go.Scatter(
    x=[str(c) for c in C_values],
    y=test_scores,
    mode='lines+markers',
    name='Test Accuracy',
    line=dict(color='red', width=2)
))

fig.update_layout(
    title="Efecto del Parámetro C en el Accuracy",
    xaxis_title="C (escala log)",
    yaxis_title="Accuracy",
    template="plotly_white",
    font=dict(size=12),
    hovermode='x unified'
)

fig.show()

print("\n💡 Observaciones:")
print(f"   • C pequeño ({C_values[0]}): {n_svs[0]} support vectors (margen amplio)")
print(f"   • C grande ({C_values[-1]}): {n_svs[-1]} support vectors (margen estrecho)")
print(f"   • C óptimo balancea bias-varianza")

In [ ]:
# Grid Search para encontrar mejores hiperparámetros
print("🔧 Buscando mejores hiperparámetros con Grid Search...\n")

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
    'kernel': ['rbf', 'poly']
}

svm_grid = SVC(random_state=42)
grid_search = GridSearchCV(svm_grid, param_grid, cv=5, scoring='accuracy', 
                          n_jobs=-1, verbose=1)

grid_search.fit(X_train_scaled, y_train)

print(f"\n✅ Mejores hiperparámetros:")
for param, value in grid_search.best_params_.items():
    print(f"   {param}: {value}")

print(f"\n📊 Mejor CV Score: {grid_search.best_score_:.4f}")
print(f"   Test Accuracy: {grid_search.score(X_test_scaled, y_test):.4f}")

### Comparación: SVM vs Otros Clasificadores

| Aspecto | SVM | Logistic Regression | Random Forest |
|---------|-----|--------------------|--------------|
| **Tipo de frontera** | Lineal o no-lineal (kernel) | Lineal | No-lineal |
| **Training Time** | O(n² a n³) | O(nd) | O(n log n × d × trees) |
| **Prediction Time** | O(n_sv × d) | O(d) | O(trees × depth) |
| **Memoria** | Solo support vectors | Todos los parámetros | Todos los árboles |
| **Escalabilidad** | Malo para n grande | Excelente | Bueno |
| **Normalización** | Necesaria | Recomendada | No necesaria |
| **Interpretabilidad** | Baja (especialmente con kernels) | Alta | Media |
| **Robustez a outliers** | Sensible | Medio | Robusto |
| **Probabilidades** | Indirectas (Platt scaling) | Nativas | Nativas |

### Cuándo usar SVM

**Usa SVM si:**
- Datos de dimensión alta (n_features >> n_samples)
- Necesitas exactitud máxima y puedes normalizar
- Los datos son limpios (pocos outliers)
- Dataset no es muy grande (< 10,000 ejemplos)
- Problemas de clasificación binaria

**NO uses SVM si:**
- Dataset enorme (millones de ejemplos)
- Muchos outliers o ruido
- Necesitas probabilidades calibradas
- Interpretabilidad es crítica
- Datos categóricos no numéricos

---
## 🎯 6. Ejercicios

### 🟢 Ejercicio 1: Efecto del Gamma en RBF

Visualiza cómo diferentes valores de gamma afectan la frontera de decisión.

In [ ]:
def ejercicio_1():
    """
    Objetivo: Entender el rol de gamma en kernel RBF
    
    Instrucciones:
    1. Genera datos con make_moons(n_samples=200, noise=0.2)
    2. Entrena SVM con kernel='rbf', C=1.0
    3. Prueba gamma = [0.1, 1, 10, 100]
    4. Para cada gamma:
       a) Calcula accuracy
       b) Cuenta support vectors
       c) Visualiza frontera de decisión
    5. Analiza: ¿cuál gamma overfits? ¿cuál underfits?
    
    Returns:
    --------
    results : dict
        {gamma: {'acc': float, 'n_sv': int}}
    """
    # TODO: Tu código aquí
    
    pass

# Descomentar para probar
# results = ejercicio_1()
# for gamma, metrics in results.items():
#     print(f"Gamma={gamma}: Acc={metrics['acc']:.3f}, SVs={metrics['n_sv']}")

### 🟡 Ejercicio 2: SVM Multiclase

SVM es naturalmente binario. Scikit-learn usa "one-vs-rest" para multiclase.

In [ ]:
def ejercicio_2():
    """
    Objetivo: Aplicar SVM a clasificación multiclase
    
    Instrucciones:
    1. Carga el dataset Iris (3 clases)
    2. Entrena SVM con diferentes kernels
    3. Para cada kernel:
       a) Calcula accuracy y matriz de confusión
       b) Identifica qué clase es más difícil de clasificar
    4. Visualiza las fronteras de decisión en 2D
       (usa solo 2 features para visualización)
    
    Returns:
    --------
    results : dict
        Diccionario con métricas por kernel
    """
    # TODO: Tu código aquí
    # Pista: usa decision_function='ovr' (one-vs-rest)
    
    pass

# Descomentar para probar
# results = ejercicio_2()
# print("\n💡 SVM puede hacer multiclase usando one-vs-rest")

### 🔴 Ejercicio 3: Implementar Kernel Personalizado

Scikit-learn permite usar kernels personalizados.

In [ ]:
def ejercicio_3():
    """
    Objetivo: Crear y usar un kernel personalizado
    
    Instrucciones:
    1. Define un kernel personalizado (ej: exponencial, Laplacian, etc.)
       Kernel Laplaciano: K(x, y) = exp(-gamma * ||x - y||_1)
    2. Usa este kernel con SVC:
       svm = SVC(kernel=tu_kernel)
    3. Compara con kernels estándar en un dataset
    4. Visualiza:
       a) Matriz de Gram (heatmap de similitudes)
       b) Frontera de decisión
       c) Distribución de support vectors
    
    Returns:
    --------
    comparison : dict
        Comparación de accuracy entre kernels
    """
    # TODO: Tu código aquí
    # Pista 1: Define una función que tome dos matrices X, Y
    #          y retorne la matriz de kernel K[i,j] = k(X[i], Y[j])
    # Pista 2: Usa metrics.pairwise para calcular distancias
    
    pass

# Descomentar para probar
# comparison = ejercicio_3()
# print("\n🎨 Kernels personalizados permiten adaptar SVM a problemas específicos")

---
## 📚 7. Resumen y Recursos

### 🎯 Puntos Clave

1. **SVM busca el margen máximo**
   - No solo separar, sino separar con la mayor "confianza"
   - Margen = distancia mínima de cualquier punto al hiperplano
   - Solo los support vectors (puntos en el margen) importan

2. **Formulación dual permite kernels**
   - Problema primal: optimizar en espacio de parámetros
   - Problema dual: optimizar en espacio de ejemplos
   - Solo aparecen productos internos → oportunidad para kernel trick

3. **Kernel trick = transformación implícita**
   - Trabajar en espacios de alta dimensión sin computar coordenadas
   - RBF kernel → espacio de dimensión infinita
   - Hace linealmente separable lo que no lo era

4. **Soft margin SVM con parámetro C**
   - Datos reales raramente son perfectamente separables
   - C controla trade-off: margen amplio vs errores bajos
   - C pequeño: margen amplio, más errores (high bias)
   - C grande: margen estrecho, menos errores (high variance)

5. **Kernels comunes:**
   - **Lineal**: datos linealmente separables, alta dimensión
   - **Polynomial**: interacciones de orden fijo
   - **RBF (Gaussian)**: universal, más usado en práctica
   - **Sigmoid**: emula redes neuronales

6. **Hiperparámetros clave:**
   - `C`: regularización (0.1 - 100)
   - `gamma` (RBF): ancho del kernel (0.001 - 1)
   - `degree` (poly): grado del polinomio (2-5)

7. **Ventajas:**
   - ✅ Efectivo en alta dimensión
   - ✅ Memory-efficient (solo SVs)
   - ✅ Versátil (diferentes kernels)
   - ✅ Funciona bien con clear margin

8. **Desventajas:**
   - ❌ Lento para datasets grandes (O(n²) a O(n³))
   - ❌ Sensible a normalización
   - ❌ Sensible a elección de kernel y params
   - ❌ No da probabilidades directamente
   - ❌ Difícil de interpretar

---

### 🔗 Recursos Adicionales

#### 📄 Papers Fundamentales

- **"A Training Algorithm for Optimal Margin Classifiers"** - Boser, Guyon, Vapnik (1992)
  - Paper original de SVM en COLT '92
  - Introduce el concepto de margen máximo

- **"Support-Vector Networks"** - Cortes & Vapnik (1995)
  - Machine Learning journal
  - Soft margin SVM y aplicaciones

- **"A Tutorial on Support Vector Machines"** - Burges (1998)
  - Excelente tutorial matemático
  - Data Mining and Knowledge Discovery
  - http://citeseer.ist.psu.edu/burges98tutorial.html

#### 📖 Libros Recomendados

- **"Learning with Kernels"** - Schölkopf & Smola (2002)
  - Libro definitivo sobre kernel methods
  - Tratamiento matemático profundo

- **"Pattern Recognition and Machine Learning"** - Bishop
  - Capítulo 7: Sparse Kernel Machines
  - Perspectiva bayesiana de SVM

- **"The Elements of Statistical Learning"** - Hastie et al.
  - Capítulo 12: Support Vector Machines
  - Comparación con otros métodos

#### 🎥 Videos Recomendados

- **MIT 6.034: Support Vector Machines** - Patrick Winston
  - Explicación intuitiva excelente
  - https://www.youtube.com/watch?v=_PwhiWxHK8o

- **StatQuest: Support Vector Machines** - Josh Starmer
  - Visualizaciones claras
  - https://www.youtube.com/watch?v=efR1C6CvhmE

- **Andrew Ng: SVMs (Stanford CS229)**
  - Derivación matemática completa

#### 💻 Documentación y Tutoriales

- [Scikit-learn: SVM Guide](https://scikit-learn.org/stable/modules/svm.html)
- [Kernel Functions for Machine Learning](http://crsouza.com/2010/03/17/kernel-functions-for-machine-learning-applications/)
- [LIBSVM](https://www.csie.ntu.edu.tw/~cjlin/libsvm/) - Librería clásica de SVM

#### 🧪 Recursos Interactivos

- [SVM Visualizer](https://cs.stanford.edu/~karpathy/svmjs/demo/) by Andrej Karpathy
- [Interactive SVM Demo](http://vision.stanford.edu/teaching/cs231n-demos/linear-classify/)

---

### 🤔 Preguntas para Reflexionar

1. **¿Por qué SVM escala mal con el tamaño del dataset?**
   - Pista: Complejidad de resolver el problema dual

2. **¿Puede un SVM con kernel RBF underfittear?**
   - Considera C muy pequeño y gamma muy pequeño

3. **¿Por qué la normalización es crítica para SVM?**
   - Piensa en la definición de margen y distancias

4. **¿Qué kernel usarías para datos de texto (bag of words)?**
   - Alta dimensionalidad, sparsity

5. **¿SVM es un modelo paramétrico o no-paramétrico?**
   - Truco: Depende de si usas representación primal o dual

---

## ➡️ Próximo Paso

En el siguiente notebook, **09. K-Means Clustering**, cambiaremos a **aprendizaje no supervisado**:

- **Clustering**: Agrupar datos sin etiquetas
- **Centroides**: Representantes de cada cluster
- **Algoritmo iterativo**: Asignar-actualizar-repetir
- **Método del codo**: Elegir K óptimo
- **Inicialización**: K-Means++ para evitar mínimos locales

**Gran cambio:** De clasificación supervisada a descubrimiento de estructura en datos sin etiquetas.

---

<div align="center">

**📐 De márgenes máximos a centroides óptimos 📐**

**Continúa con: [09. K-Means Clustering](09-kmeans.ipynb)**

[← 07. Boosting](07-boosting.ipynb) | [09. K-Means →](09-kmeans.ipynb)

</div>